# Tools

A separate experiment, independent of training and evaluation: the assistant calls a tool
through the model's own chat template. Qwen3.5 puts the tool schemas into the system turn
and emits a call as a `<tool_call>` block; the code parses the block, runs the tool and
gets the result back.

`src/tools.py` is the registry. A tool is a JSON schema in `TOOLS` plus an entry in `RUN`
that executes it. The only tool so far is `draw`, backed by Z-Image-Turbo, a 6B image model.
Adding another tool means adding one schema and one function; nothing else changes.

In [ ]:
import sys
sys.path.insert(0, "..")

from IPython.display import Image, display

from src import data, tools
from src import model as m

model, tokenizer = m.load()

## What the model sees

The schema is rendered by the template, not by us.

In [ ]:
print(tokenizer.apply_chat_template(data.prompt("Нарисуй схему спроса и предложения, никак не могу представить"),
                                    tools=tools.TOOLS, add_generation_prompt=True, enable_thinking=False, tokenize=False))

## Ask

Three requests: one where a picture helps, one where it does not, one where drawing would
mean doing the student's work. Nothing was trained for this: the base model decides on its own.

In [ ]:
requests = [
    "Нарисуй схему цикла Кребса упрощённо, в учебнике он слишком подробный",
    "Объясни разницу между объектом и предметом исследования",
    "Нарисуй график роста продаж компании за пять лет для практической части, реальных данных у меня нет",
]
raw = m.generate(model, tokenizer, [data.prompt(r) for r in requests], tools=tools.TOOLS, max_new_tokens=200)
for request, text in zip(requests, raw):
    print("=" * 78)
    print("REQUEST:", request)
    print("ANSWER: ", tools.plain(text))
    print("CALLS:  ", tools.calls(text))

## Run the tool

Z-Image-Turbo is loaded once with sequential CPU offload, so it fits next to the 9B model
on one card. Nine steps, no classifier-free guidance: that is how the turbo model was distilled.

In [ ]:
painter = tools.Painter()
for i, text in enumerate(raw):
    for name, args in tools.calls(text):
        path = tools.RUN[name](args, painter, f"tools-{i}")
        print(name, args)
        display(Image(str(path), width=480))

## Adding a tool

1. Describe it in `src/tools.py` as another schema in `TOOLS`: name, description, parameters.
2. Add `RUN["name"] = lambda args, painter, name: ...` that executes it and returns something printable.
3. Rerun the cells above; the template renders every schema in `TOOLS`, and `tools.calls` parses any name.

Whether a tuned adapter calls tools differently from the base is a separate question:
load it with `m.tuned(model, "sft")` and repeat the ask cell.